# Campus Demand Forecasting with Amazon SageMaker

This notebook trains an Amazon SageMaker **DeepAR** model to forecast occupancy rates for campus
resources (libraries, computer labs, study rooms) and generates 24-step forecasts for 5 random resources.

It uses the **SageMaker Python SDK v3**, which comes preinstalled in SageMaker Unified Studio
JupyterLab (SageMaker Distribution 4.x images).

## 1. Setup and Imports

In [ ]:
import json
import warnings
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')


def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


# This notebook is written for SageMaker Python SDK v3 (ModelTrainer)
try:
    from sagemaker.train import ModelTrainer
except ImportError as err:
    found = {n: pkg_version(n) for n in
             ['sagemaker', 'sagemaker-python-sdk', 'sagemaker-core', 'sagemaker-train']}
    raise RuntimeError(
        "This notebook needs SageMaker Python SDK v3 (sagemaker.train). "
        f"Installed SageMaker packages: {found}. "
        "Restart JupyterLab with the latest SageMaker Distribution image."
    ) from err

sdk_version = (pkg_version('sagemaker') or pkg_version('sagemaker-train')
               or pkg_version('sagemaker-python-sdk') or 'unknown')

from sagemaker_studio import Project, sqlutils
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
# ModelTrainer is imported in the version check above
from sagemaker.train.configs import Compute, InputData, OutputDataConfig, StoppingCondition

# Unified Studio project context: IAM role and project S3 location
project = Project()
role = project.iam_role
region = boto3.Session().region_name
sess = Session(boto_session=boto3.Session(region_name=region))

s3_root = project.s3.root.rstrip('/')
s3_data_path = f"{s3_root}/deepar-occupancy-forecast"

print(f"SageMaker Python SDK: {sdk_version}")
print(f"Project role: {role}")
print(f"S3 location: {s3_data_path}")
print(f"Region: {region}")

## 2. Load the Campus ML Dataset

Replace `<your_glue_database>` with the Glue database that holds your subscribed
`campus_ml_dataset` table (see the workshop instructions), then run the cell.

In [ ]:
GLUE_DATABASE = '<your_glue_database>'  # for example: glue_db_abc123xyz (see Data explorer)

if GLUE_DATABASE.startswith('<'):
    raise ValueError("Set GLUE_DATABASE to your Glue database name first (see the workshop instructions).")

query = f'SELECT * FROM "{GLUE_DATABASE}"."campus_ml_dataset"'


def athena_workgroup():
    """The project's Athena workgroup (from the project.athena connection)."""
    conn = project.connection('project.athena')
    workgroup = getattr(conn, 'athena_workgroup', None)
    if workgroup:
        return workgroup
    athena = boto3.client('athena', region_name=region)
    names = [wg['Name'] for page in athena.get_paginator('list_work_groups').paginate()
             for wg in page['WorkGroups']]
    matches = [n for n in names if project.id in n]
    if not matches:
        raise RuntimeError(f"No Athena workgroup found for project {project.id}: {names}")
    return matches[0]


def run_athena_query(sql):
    """Run a query with the Athena API and return the result as a DataFrame."""
    import time
    athena = boto3.client('athena', region_name=region)
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=athena_workgroup())['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status']
        if status['State'] in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    if status['State'] != 'SUCCEEDED':
        raise RuntimeError(f"Athena query {status['State']}: {status.get('StateChangeReason')}")
    rows = []
    for page in athena.get_paginator('get_query_results').paginate(QueryExecutionId=qid):
        rows.extend([d.get('VarCharValue') for d in r['Data']] for r in page['ResultSet']['Rows'])
    return pd.DataFrame(rows[1:], columns=rows[0])


# Try the SageMaker Unified Studio helper first, then the Athena API directly
df = None
try:
    df = sqlutils.sql(query, connection_name='project.athena')
except Exception as err:
    print(f"sqlutils.sql did not work ({err}); using the Athena API instead")
if not isinstance(df, pd.DataFrame):
    df = run_athena_query(query)

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 3. Data Preprocessing and Feature Engineering

In [ ]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Convert numeric columns (Athena can return them as strings)
for c in ['current_occupancy', 'occupancy_rate', 'total_capacity']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# One row per resource and hour, in time order
df = (df.drop_duplicates(subset=['resource_id', 'timestamp'])
        .sort_values(['resource_id', 'timestamp'])
        .reset_index(drop=True))

# Fill any missing values within each resource so the DeepAR JSON files contain no NaN
df['occupancy_rate'] = (df.groupby('resource_id')['occupancy_rate']
                          .transform(lambda s: s.interpolate(limit_direction='both')))

print(f"Rows: {len(df)}, resources: {df['resource_id'].nunique()}")
print(df.dtypes)

In [ ]:
# Lag features and rolling average, computed per resource
# (the Glue job leaves the lag columns empty, so we fill them here)
grouped = df.groupby('resource_id')['occupancy_rate']
df['occupancy_rate_lag_1h'] = grouped.shift(1)
df['occupancy_rate_lag_24h'] = grouped.shift(24)
df['occupancy_rate_lag_168h'] = grouped.shift(168)  # 1 week
df['occupancy_rate_avg_24h'] = grouped.transform(lambda s: s.rolling(window=24, min_periods=1).mean())

# Fill the first periods of each resource, where no history exists yet
lag_cols = ['occupancy_rate_lag_1h', 'occupancy_rate_lag_24h', 'occupancy_rate_lag_168h', 'occupancy_rate_avg_24h']
df[lag_cols] = df[lag_cols].fillna(0)

print("Lag features created. Sample:")
print(df[['resource_id', 'timestamp', 'occupancy_rate'] + lag_cols].head(30))

In [ ]:
# Encode categorical features
categorical_cols = ['day_of_week', 'is_exam_period', 'is_peak_hour', 'resource_type', 'location']
label_encoders = {}

for cat_col in categorical_cols:
    if cat_col in df.columns:
        le = LabelEncoder()
        df[f'{cat_col}_encoded'] = le.fit_transform(df[cat_col].astype(str))
        label_encoders[cat_col] = le
        print(f"{cat_col}: {list(le.classes_)}")

print("\nCategorical encoding complete.")

## 4. Prepare Data for DeepAR

In [ ]:
# DeepAR expects one JSON object per time series (JSON Lines)
prediction_length = 24  # forecast the next 24 time steps
context_length = 168    # look back over the previous 168 time steps

dynamic_cols = ['day_of_week_encoded', 'is_exam_period_encoded', 'is_peak_hour_encoded']
static_cat_cols = ['resource_type_encoded', 'location_encoded']

resources = df['resource_id'].unique()
print(f"Total resources: {len(resources)}")

train_data, test_data, resource_mapping = [], [], []

for resource in resources:
    resource_df = df[df['resource_id'] == resource].sort_values('timestamp')

    # Skip resources without enough history
    if len(resource_df) < context_length + prediction_length:
        continue

    target = [float(v) for v in resource_df['occupancy_rate']]
    start_time = resource_df['timestamp'].min().strftime('%Y-%m-%d %H:%M:%S')
    dynamic_feat = [[int(v) for v in resource_df[c]] for c in dynamic_cols]
    cat_features = [int(resource_df[c].iloc[0]) for c in static_cat_cols]

    # Train: everything except the last prediction_length steps
    train_data.append({
        "start": start_time,
        "target": target[:-prediction_length],
        "dynamic_feat": [f[:-prediction_length] for f in dynamic_feat],
        "cat": cat_features,
    })
    # Test: the full series, used to evaluate the held-out last steps
    test_data.append({
        "start": start_time,
        "target": target,
        "dynamic_feat": dynamic_feat,
        "cat": cat_features,
    })
    resource_mapping.append(resource)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

In [ ]:
# Save data to JSON Lines format
def write_jsonlines(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, allow_nan=False) + '\n')

write_jsonlines(train_data, 'train.json')
write_jsonlines(test_data, 'test.json')
print("Data files created: train.json, test.json")

In [ ]:
# Upload to the project S3 location
bucket, prefix = s3_data_path.replace('s3://', '').split('/', 1)
s3 = boto3.client('s3', region_name=region)
s3.upload_file('train.json', bucket, f"{prefix}/train/train.json")
s3.upload_file('test.json', bucket, f"{prefix}/test/test.json")

s3_train_path = f"{s3_data_path}/train/"
s3_test_path = f"{s3_data_path}/test/"
print(f"Train data uploaded to: {s3_train_path}")
print(f"Test data uploaded to: {s3_test_path}")

## 5. Train the DeepAR Model

In [ ]:
# Built-in DeepAR container image for this region
image_uri = image_uris.retrieve(framework='forecasting-deepar', region=region)
print(f"DeepAR image: {image_uri}")

In [ ]:
hyperparameters = {
    'time_freq': 'H',  # hourly observations
    'epochs': 100,
    'early_stopping_patience': 10,
    'mini_batch_size': 32,
    'learning_rate': 0.001,
    'context_length': context_length,
    'prediction_length': prediction_length,
    'num_cells': 40,
    'num_layers': 3,
    'likelihood': 'gaussian',
    'dropout_rate': 0.1,
    'embedding_dimension': 10,
    'num_dynamic_feat': len(dynamic_cols),
    'cardinality': json.dumps([len(label_encoders['resource_type'].classes_),
                               len(label_encoders['location'].classes_)]),
}
# DeepAR expects every hyperparameter value as a string
hyperparameters = {k: str(v) for k, v in hyperparameters.items()}

model_trainer = ModelTrainer(
    training_image=image_uri,
    role=role,
    sagemaker_session=sess,
    base_job_name='campus-deepar',
    compute=Compute(instance_type='ml.c5.2xlarge', instance_count=1),
    hyperparameters=hyperparameters,
    output_data_config=OutputDataConfig(s3_output_path=f"{s3_data_path}/output"),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
)

print("Model trainer configured.")

In [ ]:
# Train the model (takes about 5-10 minutes)
model_trainer.train(
    input_data_config=[
        InputData(channel_name='train', data_source=s3_train_path),
        InputData(channel_name='test', data_source=s3_test_path),
    ],
    wait=True,
    logs=True,
)

training_job = model_trainer._latest_training_job
training_job.refresh()
model_data = training_job.model_artifacts.s3_model_artifacts
print(f"Training job: {training_job.training_job_name}")
print(f"Model artifact: {model_data}")

## 6. Deploy the Model to a Real-time Endpoint

In [ ]:
# Takes about 5-8 minutes
name = f"campus-deepar-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

model = Model.create(
    model_name=name,
    primary_container=ContainerDefinition(image=image_uri, model_data_url=model_data),
    execution_role_arn=role,
)
endpoint_config = EndpointConfig.create(
    endpoint_config_name=name,
    production_variants=[ProductionVariant(
        variant_name='AllTraffic',
        model_name=name,
        initial_instance_count=1,
        instance_type='ml.m5.xlarge',
    )],
)
endpoint = Endpoint.create(endpoint_name=name, endpoint_config_name=name)
endpoint.wait_for_status('InService')

print(f"Model deployed to endpoint: {endpoint.endpoint_name}")

## 7. Generate 24-step Forecasts for 5 Random Resources

In [ ]:
# Select 5 random resources
np.random.seed(42)
random_indices = np.random.choice(len(test_data), size=min(5, len(test_data)), replace=False)
random_test_samples = [(test_data[i], resource_mapping[i]) for i in random_indices]

print(f"Selected {len(random_test_samples)} random resources for prediction")
print(f"Resource IDs: {[r[1] for r in random_test_samples]}")

In [ ]:
def predict(instance):
    request = {
        "instances": [instance],
        "configuration": {
            "num_samples": 100,
            "output_types": ["mean", "quantiles"],
            "quantiles": ["0.1", "0.5", "0.9"],
        },
    }
    response = endpoint.invoke(
        body=json.dumps(request),
        content_type='application/json',
        accept='application/json',
    )
    body = response.body.read() if hasattr(response.body, 'read') else response.body
    return json.loads(body)["predictions"][0]

predictions = []
for sample, resource_id in random_test_samples:
    # History = all but the last prediction_length steps.
    # dynamic_feat must also cover the forecast window, so we pass the full feature series.
    instance = {
        "start": sample["start"],
        "target": sample["target"][:-prediction_length],
        "dynamic_feat": sample["dynamic_feat"],
        "cat": sample["cat"],
    }
    result = predict(instance)
    predictions.append({
        "resource_id": resource_id,
        "actual": sample["target"][-prediction_length:],
        "predicted_mean": result["mean"],
        "predicted_quantiles": result["quantiles"],
    })

print(f"Generated predictions for {len(predictions)} resources")

## 8. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(len(predictions), 1, figsize=(15, 4 * len(predictions)))
axes = np.atleast_1d(axes)

for ax, pred in zip(axes, predictions):
    steps = list(range(1, prediction_length + 1))
    ax.plot(steps, pred["actual"], label='Actual', marker='o', linewidth=2)
    ax.plot(steps, pred["predicted_mean"], label='Predicted Mean', marker='s', linewidth=2, linestyle='--')
    ax.fill_between(steps, pred["predicted_quantiles"]["0.1"], pred["predicted_quantiles"]["0.9"],
                    alpha=0.3, label='80% Confidence Interval')
    ax.set_xlabel('Steps Ahead (hours of operation)')
    ax.set_ylabel('Occupancy Rate')
    ax.set_title(f'Resource {pred["resource_id"]}: 24-step Occupancy Rate Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('occupancy_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualization saved as 'occupancy_forecasts.png'")

## 9. Calculate Prediction Metrics

In [ ]:
metrics_summary = []
for pred in predictions:
    actual = np.array(pred["actual"])
    predicted = np.array(pred["predicted_mean"])
    metrics_summary.append({
        'Resource': pred["resource_id"],
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
        'MAPE (%)': np.mean(np.abs((actual - predicted) / np.maximum(np.abs(actual), 1e-10))) * 100,
    })

metrics_df = pd.DataFrame(metrics_summary)
print("Prediction Metrics:")
print(metrics_df.to_string(index=False))
print("\nAverage Metrics:")
print(metrics_df[['MAE', 'RMSE', 'R2', 'MAPE (%)']].mean().to_string())

## 10. Clean Up

The endpoint is billed while it runs. Run this cell when you are done.

In [ ]:
endpoint.delete()
endpoint_config.delete()
model.delete()
print("Endpoint, endpoint configuration and model deleted")

## Summary

This notebook:
1. Loaded the subscribed campus ML dataset with Amazon Athena
2. Created lag features (1h, 24h, 168h) and rolling averages
3. Encoded categorical features for machine learning
4. Prepared data in DeepAR JSON Lines format
5. Trained a DeepAR forecasting model with the SageMaker Python SDK v3 `ModelTrainer`
6. Deployed the model and generated 24-step forecasts for 5 random resources
7. Visualized predictions with confidence intervals
8. Calculated prediction accuracy metrics